In [1]:
import os
import sys
import hashlib

import numpy as np
import pandas as pd
from scipy.stats import bootstrap

sys.path.insert(0, os.path.abspath(".."))
from lib import census as lc
from lib import io as lio

In [2]:
year = 2018

In [3]:
# these two partition the 2018 pums
pums = pd.read_parquet("../data/pums/pums_100_2018.parquet")
pums

,REF_INDEX,SEC_INDEX,NUM_CHILDREN_UNDER_6,NUM_CHILDREN_6_TO_17,SIZE,NUM_IN_IF,NUM_WORKING,PERWT,CHOSEN,ORIGIN,...,AAPI_REF,OTHER_RACE_REF,RACE_ETHNICITY_REF,LATINO_SEC,WHITE_SEC,BLACK_SEC,INDIAN_SEC,AAPI_SEC,OTHER_RACE_SEC,RACE_ETHNICITY_SEC
UNIT,,,,,,,,,,,,,,,,,,,,,
2018000277858_P,2018000277858001,2018000277858002,0,0,2,0,0,46.0,4400104,4400100,...,0,0,1,0,1,0,0,0,0,1
2018001331204_I2,2018001331204002,2018001331204002,0,0,1,1,1,215.0,1703523,1703400,...,0,0,99,1,0,0,0,0,0,99
2018001188817_P,2018001188817001,2018001188817002,0,0,2,2,2,59.0,4703100,4703100,...,0,0,1,0,1,0,0,0,0,1
2018001182700_P,2018001182700001,2018001182700001,0,0,1,1,1,112.0,1702400,1702400,...,0,0,1,0,1,0,0,0,0,1
2018000271732_I3,2018000271732003,2018000271732003,0,0,1,1,1,125.0,1205301,1205301,...,0,0,1,0,1,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018000947916_P,2018000947916001,2018000947916001,0,0,1,1,1,67.0,1203102,1203100,...,0,0,2,0,0,1,0,0,0,2
2018000950420_P,2018000950420001,2018000950420002,0,0,2,2,2,88.0,2401006,2401000,...,1,0,4,0,0,0,0,1,0,4
2018010134587_I1,2018010134587001,2018010134587001,0,0,1,0,0,81.0,2504303,2500390,...,0,0,1,0,1,0,0,0,0,1


In [4]:
def filter_cols(col):
    return col.startswith("REPWTP") or col in [
        "CBSERIAL",
        "PERNUM",
    ]


replicate_weights = pd.read_csv(
    "../data/pums/us/pums_2018_raw.csv", usecols=filter_cols
)
replicate_weights["PID"] = (
    replicate_weights["CBSERIAL"] * 1_000 + replicate_weights["PERNUM"]
)
assert replicate_weights["PID"].is_unique
replicate_weights = replicate_weights.set_index("PID")
replicate_weights

,CBSERIAL,PERNUM,REPWTP,REPWTP1,REPWTP2,REPWTP3,REPWTP4,REPWTP5,REPWTP6,REPWTP7,...,REPWTP71,REPWTP72,REPWTP73,REPWTP74,REPWTP75,REPWTP76,REPWTP77,REPWTP78,REPWTP79,REPWTP80
PID,,,,,,,,,,,,,,,,,,,,,
2018010000049001,2018010000049,1,1,72,74,139,145,7,4,144,...,140,74,73,7,76,75,80,74,7,72
2018010000058001,2018010000058,1,1,6,6,146,77,80,6,149,...,76,78,7,76,80,78,7,147,150,75
2018010000219001,2018010000219,1,1,210,122,116,235,124,19,16,...,117,121,123,205,208,218,120,19,123,18
2018010000246001,2018010000246,1,1,7,44,45,5,8,8,6,...,43,76,79,77,80,44,46,82,81,8
2018010000251001,2018010000251,1,1,15,18,30,15,30,16,16,...,4,2,29,17,15,28,17,30,15,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018001400326004,2018001400326,4,1,25,170,79,122,149,104,143,...,96,28,98,111,88,27,30,92,73,112
2018001400326005,2018001400326,5,1,24,170,80,122,149,102,144,...,96,28,97,111,88,28,30,92,72,110
2018001400502001,2018001400502,1,1,19,45,17,68,34,49,109,...,44,18,34,85,18,15,85,47,31,22


In [5]:
pums = pums.merge(replicate_weights, how="left", left_on="REF_INDEX", right_index=True)
pums

,REF_INDEX,SEC_INDEX,NUM_CHILDREN_UNDER_6,NUM_CHILDREN_6_TO_17,SIZE,NUM_IN_IF,NUM_WORKING,PERWT,CHOSEN,ORIGIN,...,REPWTP71,REPWTP72,REPWTP73,REPWTP74,REPWTP75,REPWTP76,REPWTP77,REPWTP78,REPWTP79,REPWTP80
UNIT,,,,,,,,,,,,,,,,,,,,,
2018000277858_P,2018000277858001,2018000277858002,0,0,2,0,0,46.0,4400104,4400100,...,14,45,68,14,14,43,50,88,44,47
2018001331204_I2,2018001331204002,2018001331204002,0,0,1,1,1,215.0,1703523,1703400,...,415,66,196,219,344,72,228,239,82,71
2018001188817_P,2018001188817001,2018001188817002,0,0,2,2,2,59.0,4703100,4703100,...,18,87,59,70,105,55,64,123,59,56
2018001182700_P,2018001182700001,2018001182700001,0,0,1,1,1,112.0,1702400,1702400,...,30,124,192,117,113,194,115,37,39,38
2018000271732_I3,2018000271732003,2018000271732003,0,0,1,1,1,125.0,1205301,1205301,...,118,36,126,35,217,92,46,38,110,105
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018000947916_P,2018000947916001,2018000947916001,0,0,1,1,1,67.0,1203102,1203100,...,82,115,60,64,98,64,19,17,19,102
2018000950420_P,2018000950420001,2018000950420002,0,0,2,2,2,88.0,2401006,2401000,...,95,89,29,115,96,146,136,81,146,75
2018010134587_I1,2018010134587001,2018010134587001,0,0,1,0,0,81.0,2504303,2500390,...,81,84,79,12,80,79,148,13,148,79


In [ ]:
# per-geography (PUMA / MIGPUMA) tables: ACS + LODES + CBSA + weather + CBP establishment
# counts joined per area, with the numeric CBSA type/name encoding (factorized on the PUMA
# side so the two are comparable) and the derived per-capita / NAICS job-share columns.
# Same tables create_estdata.ipynb builds its .ORIG and per-alternative blocks from.
puma_data, migpuma_data = lc.load_geo_tables("../data", year)

# PUMA<->MIGPUMA equivalency table (already indexed by PUMA, State already zero-padded
# as it's read as dtype=str)
puma_migpuma = lio.load_puma_migpuma(
    "../data/geometry/equivalencies/puma_migpuma_2010.csv"
)

In [9]:
# current origin(MIGPUMA)->destination(PUMA) distance/time matrices
distance_matrix = lio.load_distance_matrix(
    "../data/distances/puma_migpuma_distance_matrix.csv", zfill=7
)
time_matrix = lio.load_distance_matrix(
    "../data/distances/puma_migpuma_time_matrix.csv", zfill=7
)
alt_pool = np.array(distance_matrix.columns, dtype="<U7")
len(alt_pool)

2336

In [13]:
# Five of the stay-utility regressors are not plain MIGPUMA columns: they are per-unit
# gathers of "this unit's own category's share at its origin", so
# compare_means(level="migpuma") cannot reach them. Materialise them on `pums` here --
# built the same way create_estdata.ipynb builds the corresponding .ORIG columns -- and
# compare them at level="pums".
UNIT_SUFFIXES = lc.UNIT_SUFFIXES
row = np.arange(len(pums))


def gather(block, col_names):
    """Per-row pick of one column of `block` (already aligned to `pums`), named per row
    by `col_names` -- the paired (row, col) gather create_estdata uses throughout."""
    pos = block.columns.get_indexer(col_names)
    assert (pos >= 0).all(), "some unit's category has no matching column"
    return block.to_numpy(float)[row, pos]


# stay_med_earnings_10k_{no_degree,degree}: the origin's median earnings for the unit's
# own education bracket. EDU_* partitions the units, so there is no missing case.
own_edu_col_name = np.select(
    [pums[flag] == 1 for flag in lc.EDU_EARNINGS_RAW_COLS],
    list(lc.EDU_EARNINGS_RAW_COLS.values()),
    default="",
)
pums["OWN_EARNINGS_10K_BY_EDU"] = gather(
    migpuma_data[list(lc.EDU_EARNINGS_RAW_COLS.values())].reindex(pums["ORIGIN"]),
    own_edu_col_name,
)

# proportion_same_race_*: the origin's population share in each member's own category.
# Keyed on IPUMS RACE with Hispanic overriding race, NOT ACS RAC1P -- see lib/census.py.
race_block = migpuma_data[list(dict.fromkeys(lc.RACE_COLUMNS.values()))].reindex(
    pums["ORIGIN"]
)
for sfx in UNIT_SUFFIXES:
    pums[f"OWN_RACE_ETH_PROP_{sfx}"] = gather(
        race_block, pums[f"RACE_ETHNICITY_{sfx}"].map(lc.RACE_COLUMNS).to_numpy()
    )

# proportion_same_naics_*: the origin's share of jobs in each member's own
# skill/credential group.
sector_to_group_col = {
    sector: f"NAICS_GROUP_PROP_{group}"
    for group, sectors in lc.NAICS_GROUPS.items()
    for sector in sectors
}
group_block = migpuma_data[lc.NAICS_GROUP_PROP_COLUMNS].reindex(pums["ORIGIN"])
for sfx in UNIT_SUFFIXES:
    sector = pums[f"INDNAICS_{sfx}"].str[0:2].map(lc.NAICS_SECTOR_PREFIXES)
    # active-duty military is excluded from PUB, as in create_estdata
    sector = sector.where(~pums[f"IN_MILITARY_{sfx}"].astype(bool))
    group_col = sector.map(sector_to_group_col)
    # NaN (not NaN->0 as in create_estdata) for members with no group: _values_and_weights
    # drops non-finite values, so the mean is over members who actually have a job group
    # rather than being pulled toward 0 by the unemployed/NILF/military.
    pums[f"OWN_NAICS_GROUP_PROP_{sfx}"] = np.where(
        group_col.isna(),
        np.nan,
        gather(
            group_block,
            group_col.fillna(lc.NAICS_GROUP_PROP_COLUMNS[0]).to_numpy(),
        ),
    )

# origin MIGPUMA's population share in the unit's own age bracket (AGE_UNDER_18 is
# empty -- both members are required to be 18+ upstream)
AGE_BRACKET_COLS = [
    "Proportion of people 18-34",
    "Proportion of people 35-64",
    "Proportion of people 65+",
]
own_age_col_name = np.select(
    [pums[f].to_numpy() == 1 for f in ["AGE_18_34", "AGE_35_64", "AGE_OVER_65"]],
    AGE_BRACKET_COLS,
    default="",
)
assert (own_age_col_name != "").all(), "AGE_* brackets do not partition the units"
pums["OWN_AGE_PROP.ORIG"] = gather(
    migpuma_data[AGE_BRACKET_COLS].reindex(pums["ORIGIN"]), own_age_col_name
)

# indicators not carried on pums / migpuma_data
for sfx in UNIT_SUFFIXES:
    sector = pums[f"INDNAICS_{sfx}"].str[0:2].map(lc.NAICS_SECTOR_PREFIXES)
    sector = sector.where(~pums[f"IN_MILITARY_{sfx}"].astype(bool))
    for group, sectors in lc.NAICS_GROUPS.items():
        pums[f"NAICS_{group}_{sfx}"] = sector.isin(sectors).astype(int)

migpuma_data["IS_T34"] = (migpuma_data["TYPE_NUM"] == 0).astype(int)
migpuma_data["IS_METRO"] = (migpuma_data["TYPE_NUM"] == 1).astype(int)
migpuma_data["NOT_MSA"] = (migpuma_data["TYPE_NUM"] == 2).astype(int)

In [14]:
movers = pums[pums["STAY"] == 0].copy()
stayers = pums[pums["STAY"] == 1].copy()

In [15]:
for c in movers.columns:
    print(c)

REF_INDEX
SEC_INDEX
NUM_CHILDREN_UNDER_6
NUM_CHILDREN_6_TO_17
SIZE
NUM_IN_IF
NUM_WORKING
PERWT
CHOSEN
ORIGIN
STAY
STATEFIP
ORIGIN_STATE
GRADEATT_REF
GRADEATT_SEC
RACE_REF
RACE_SEC
HISPAN_REF
HISPAN_SEC
BPL_REF
BPL_SEC
CITIZEN_REF
CITIZEN_SEC
WORKING_REF
WORKING_SEC
INDNAICS_REF
INDNAICS_SEC
CLASSWKR_REF
CLASSWKR_SEC
AGE_REF
AGE_SEC
MARST_REF
MARST_SEC
DIVINYR_REF
DIVINYR_SEC
WIDINYR_REF
WIDINYR_SEC
MARRINYR_REF
MARRINYR_SEC
VETSTATD_REF
VETSTATD_SEC
EMPSTAT_REF
EMPSTAT_SEC
RACAMIND_REF
RACAMIND_SEC
RACASIAN_REF
RACASIAN_SEC
RACBLK_REF
RACBLK_SEC
RACPACIS_REF
RACPACIS_SEC
RACWHT_REF
RACWHT_SEC
RACOTHER_REF
RACOTHER_SEC
EDUC_REF
EDUC_SEC
EDUCD_REF
EDUCD_SEC
RELATE_REF
RELATE_SEC
PAIRED_UNIT
CHILD_UNDER_6
CHILD_6_TO_17
CHILD
NUM_CHILDREN
WORK2
WORK1
PAIR_WORK1
SINGLE_UNIT_WITH_CHILD
MAX_EDUC
MAX_EDUCD
EDU_NOHIGH
EDU_ONLY_HIGH
EDU_SOME_COLLEGE
EDU_ONLY_BACHELORS
EDU_GRADUATE_DEG
EDU_BACHELORS_OR_HIGHER
EDU_HAS_DEGREE
EDU_NO_DEGREE
EDU_HIGH_BUT_NOT_BACHELORS
MEAN_AGE
AGE_UNDER_18
AGE_18_34


In [16]:
WEIGHT_COLUMNS = ["PERWT"] + [f"REPWTP{replicate}" for replicate in range(1, 81)]
N_WEIGHT_COLUMNS = len(WEIGHT_COLUMNS)


def _values_and_weights(group, column, level):
    """-> (values, weights) with weights of shape (n_units, 81).

    Column 0 of weights is the full-sample weight; columns 1-80 are replicates.
    """
    person_weights = group[WEIGHT_COLUMNS].to_numpy(float)

    if level == "pums":
        person_values = group[column].to_numpy(float)
        is_present = np.isfinite(person_values)
        return person_values[is_present], person_weights[is_present]
    elif level == "migpuma":
        key_to_geography = "ORIGIN"
        data = migpuma_data
    elif level == "puma":
        key_to_geography = "CHOSEN"
        data = puma_data
    else:
        raise ValueError("unknown level")

    # migpuma/puma: one row per origin area, person weights summed within area
    area_codes, area_index = pd.factorize(group[key_to_geography], sort=True)
    area_weights = np.stack(
        [
            np.bincount(
                area_codes, weights=person_weights[:, i], minlength=len(area_index)
            )
            for i in range(N_WEIGHT_COLUMNS)
        ],
        axis=1,
    )
    area_values = data[column].reindex(area_index).to_numpy(float)
    return area_values, area_weights


def _weighted_totals(group, column, level):
    """-> (sum(w*v), sum(w*v**2), sum(w)), each of shape (81,).

    Returned as raw totals rather than means so the POOLED (movers + stayers) mean and
    spread can be recovered by simple addition -- pooling two groups' totals is exact,
    and it avoids materialising the 81 weight columns a third time.
    """
    values, weights = _values_and_weights(group, column, level)
    return values @ weights, (values**2) @ weights, weights.sum(axis=0)


def _weighted_means(group, column, level):
    """-> (81,) weighted means, one per weight column."""
    # basically compute a weighted average over the column using 81 different weights
    total, _, weight = _weighted_totals(group, column, level)
    return total / weight


def _replicate_se(estimates):
    """ACS successive-difference replicate standard error for an (81,) estimate array.

    per https://www2.census.gov/programs-surveys/acs/methodology/design_and_methodology/2014/acs_design_methodology_report_2014.pdf
    this is how we can estimate variances
    """
    return np.sqrt(4 / 80 * ((estimates[1:] - estimates[0]) ** 2).sum())


def compare_means(movers, stayers, column, level="pums"):
    mover_total, mover_square, mover_weight = _weighted_totals(movers, column, level)
    stayer_total, stayer_square, stayer_weight = _weighted_totals(
        stayers, column, level
    )

    mover_means = mover_total / mover_weight
    stayer_means = stayer_total / stayer_weight

    # difference taken within each replicate, so the covariance is retained
    differences = mover_means - stayer_means

    # Pooled mean and SD of the variable itself, recovered from the two groups' totals.
    # These are the yardsticks the raw difference is reported against: a difference is
    # only "large" relative to the level (% of mean) or the spread (/ SD) of what is
    # being differenced.
    pooled_weight = mover_weight + stayer_weight
    pooled_means = (mover_total + stayer_total) / pooled_weight
    pooled_variance = (mover_square + stayer_square) / pooled_weight - pooled_means**2
    pooled_sd = np.sqrt(np.maximum(pooled_variance, 0.0))

    difference = differences[0]
    standard_error = _replicate_se(differences)

    return {
        "variable": column,
        "level": level,
        "mover_mean": mover_means[0],
        "stayer_mean": stayer_means[0],
        "pooled_mean": pooled_means[0],
        "pooled_sd": pooled_sd[0],
        "diff": difference,
        "diff_se": standard_error,
        "diff_ci": (
            difference - 1.96 * standard_error,
            difference + 1.96 * standard_error,
        ),
        # difference in units of the variable's own between-unit spread
        "std_diff": difference / pooled_sd[0] if pooled_sd[0] > 0 else np.nan,
    }


def print_result(result):
    low, high = result["diff_ci"]
    print(f"{result['variable']} ({result['level']})")
    print(f"  Mover mean (wtd):        {result['mover_mean']:,.6f}")
    print(f"  Stayer mean (wtd):       {result['stayer_mean']:,.6f}")
    print(f"  Overall mean (wtd):      {result['pooled_mean']:,.6f}")
    print(f"  Overall SD (wtd):        {result['pooled_sd']:,.6f}")
    print(f"  Diff (Movers - Stayers): {result['diff']:.6f}")
    print(f"  SE:                      {result['diff_se']:.6f}")
    print(f"  95% CI:                  [{low:.6f}, {high:.6f}]")
    print(f"  Diff / SD:               {result['std_diff']:+.6f}\n")


In [17]:
print(pums["PERWT"].sum())
print(movers["PERWT"].sum())
print(movers["PERWT"].sum() / pums["PERWT"].sum())

179587984.0
10316808.0
0.05744709512413704


In [18]:
result = compare_means(
    movers,
    stayers,
    "MEAN_AGE",
    "pums",
)
print_result(result)

MEAN_AGE (pums)
  Mover mean (wtd):        35.922301
  Stayer mean (wtd):       47.047567
  Overall mean (wtd):      46.408453
  Overall SD (wtd):        19.215057
  Diff (Movers - Stayers): -11.125266
  SE:                      0.076430
  95% CI:                  [-11.275070, -10.975463]
  Diff / SD:               -0.578987



In [19]:
result = compare_means(
    movers,
    stayers,
    "Median Gross Rent.Median Gross Rent.SE_A18009_001",
    "migpuma",
)
print_result(result)

Median Gross Rent.Median Gross Rent.SE_A18009_001 (migpuma)
  Mover mean (wtd):        1,126.189528
  Stayer mean (wtd):       1,117.453832
  Overall mean (wtd):      1,117.955672
  Overall SD (wtd):        347.694994
  Diff (Movers - Stayers): 8.735696
  SE:                      1.507902
  95% CI:                  [5.780208, 11.691184]
  Diff / SD:               +0.025125



In [20]:
result = compare_means(
    movers,
    stayers,
    "Median House Value for All Owner-Occupied Housing Units.Median Value.SE_A10036_001",
    "migpuma",
)
print_result(result)

Median House Value for All Owner-Occupied Housing Units.Median Value.SE_A10036_001 (migpuma)
  Mover mean (wtd):        300,680.107540
  Stayer mean (wtd):       298,069.212035
  Overall mean (wtd):      298,219.200398
  Overall SD (wtd):        209,036.915513
  Diff (Movers - Stayers): 2610.895504
  SE:                      915.528648
  95% CI:                  [816.459355, 4405.331653]
  Diff / SD:               +0.012490



In [21]:
result = compare_means(
    movers,
    stayers,
    "Density",
    "migpuma",
)
print_result(result)

Density (migpuma)
  Mover mean (wtd):        1,914.366906
  Stayer mean (wtd):       1,844.213395
  Overall mean (wtd):      1,848.243510
  Overall SD (wtd):        4,995.403673
  Diff (Movers - Stayers): 70.153512
  SE:                      23.789942
  95% CI:                  [23.525226, 116.781797]
  Diff / SD:               +0.014044



In [22]:
# area-level ACS variable via migpuma_data, e.g. house value
result = compare_means(
    movers,
    stayers,
    "Unemployment rate",
    "migpuma",
)
print_result(result)

Unemployment rate (migpuma)
  Mover mean (wtd):        0.049092
  Stayer mean (wtd):       0.050200
  Overall mean (wtd):      0.050137
  Overall SD (wtd):        0.015033
  Diff (Movers - Stayers): -0.001108
  SE:                      0.000073
  95% CI:                  [-0.001250, -0.000966]
  Diff / SD:               -0.073702



In [23]:
# area-level ACS variable via migpuma_data, e.g. house value
result = compare_means(
    movers,
    stayers,
    "House vacancy proportion",
    "migpuma",
)
print_result(result)

House vacancy proportion (migpuma)
  Mover mean (wtd):        0.114054
  Stayer mean (wtd):       0.115857
  Overall mean (wtd):      0.115753
  Overall SD (wtd):        0.066207
  Diff (Movers - Stayers): -0.001803
  SE:                      0.000315
  95% CI:                  [-0.002421, -0.001185]
  Diff / SD:               -0.027227



In [24]:
# stay_med_earnings_10k_no_degree / stay_med_earnings_10k_degree
result = compare_means(
    movers,
    stayers,
    "OWN_EARNINGS_10K_BY_EDU",
    "pums",
)
print_result(result)

OWN_EARNINGS_10K_BY_EDU (pums)
  Mover mean (wtd):        4.437756
  Stayer mean (wtd):       4.269342
  Overall mean (wtd):      4.279017
  Overall SD (wtd):        1.675314
  Diff (Movers - Stayers): 0.168415
  SE:                      0.006938
  95% CI:                  [0.154816, 0.182014]
  Diff / SD:               +0.100527



In [25]:
# stay_alt_commute
result = compare_means(
    movers,
    stayers,
    "Proportion alternative commute",
    "migpuma",
)
print_result(result)

Proportion alternative commute (migpuma)
  Mover mean (wtd):        0.046617
  Stayer mean (wtd):       0.044906
  Overall mean (wtd):      0.045005
  Overall SD (wtd):        0.088431
  Diff (Movers - Stayers): 0.001710
  SE:                      0.000401
  95% CI:                  [0.000924, 0.002497]
  Diff / SD:               +0.019341



In [26]:
# proportion_same_age_18_34 / _35_64 / _65_plus
result = compare_means(
    movers,
    stayers,
    "OWN_AGE_PROP.ORIG",
    "pums",
)
print_result(result)

OWN_AGE_PROP.ORIG (pums)
  Mover mean (wtd):        0.276193
  Stayer mean (wtd):       0.292244
  Overall mean (wtd):      0.291322
  Overall SD (wtd):        0.095001
  Diff (Movers - Stayers): -0.016051
  SE:                      0.000318
  95% CI:                  [-0.016675, -0.015427]
  Diff / SD:               -0.168959



In [27]:
# proportion_hh_with_children_if_have_children
result = compare_means(
    movers[movers["CHILD"] == 1],
    stayers[stayers["CHILD"] == 1],
    "Proportion of households with children",
    "migpuma",
)
print_result(result)

Proportion of households with children (migpuma)
  Mover mean (wtd):        0.305632
  Stayer mean (wtd):       0.307605
  Overall mean (wtd):      0.307523
  Overall SD (wtd):        0.050207
  Diff (Movers - Stayers): -0.001973
  SE:                      0.000539
  95% CI:                  [-0.003030, -0.000917]
  Diff / SD:               -0.039304



In [28]:
# proportion_college_if_in_college
result = compare_means(
    movers[movers["IN_COLLEGE"] == 1],
    stayers[stayers["IN_COLLEGE"] == 1],
    "Proportion of people in college",
    "migpuma",
)
print_result(result)

Proportion of people in college (migpuma)
  Mover mean (wtd):        0.070908
  Stayer mean (wtd):       0.075949
  Overall mean (wtd):      0.075305
  Overall SD (wtd):        0.033810
  Diff (Movers - Stayers): -0.005041
  SE:                      0.000324
  95% CI:                  [-0.005676, -0.004406]
  Diff / SD:               -0.149105



In [29]:
# proportion_foreign_if_foreign
result = compare_means(
    movers[movers["FOREIGN_BORN"] == 1],
    stayers[stayers["FOREIGN_BORN"] == 1],
    "Proportion foreign born",
    "migpuma",
)
print_result(result)

Proportion foreign born (migpuma)
  Mover mean (wtd):        0.197504
  Stayer mean (wtd):       0.226740
  Overall mean (wtd):      0.225476
  Overall SD (wtd):        0.122012
  Diff (Movers - Stayers): -0.029236
  SE:                      0.001521
  95% CI:                  [-0.032218, -0.026254]
  Diff / SD:               -0.239614



In [30]:
# median_travel_time
result = compare_means(
    movers,
    stayers,
    "Median travel time",
    "migpuma",
)
print_result(result)

Median travel time (migpuma)
  Mover mean (wtd):        25.138084
  Stayer mean (wtd):       25.285932
  Overall mean (wtd):      25.277439
  Overall SD (wtd):        5.936233
  Diff (Movers - Stayers): -0.147849
  SE:                      0.026377
  95% CI:                  [-0.199548, -0.096150]
  Diff / SD:               -0.024906



In [31]:
# proportion_also_mil
result = compare_means(
    movers[movers["IN_MILITARY"] == 1],
    stayers[stayers["IN_MILITARY"] == 1],
    "Proportion of people in military",
    "migpuma",
)
print_result(result)

Proportion of people in military (migpuma)
  Mover mean (wtd):        0.028939
  Stayer mean (wtd):       0.055216
  Overall mean (wtd):      0.046914
  Overall SD (wtd):        0.056973
  Diff (Movers - Stayers): -0.026277
  SE:                      0.001569
  95% CI:                  [-0.029352, -0.023202]
  Diff / SD:               -0.461211



In [32]:
# proportion_same_naics_* (reference person)
result = compare_means(
    movers[~movers["OWN_NAICS_GROUP_PROP_REF"].isna()],
    stayers[~stayers["OWN_NAICS_GROUP_PROP_REF"].isna()],
    "OWN_NAICS_GROUP_PROP_REF",
    "pums",
)
print_result(result)

OWN_NAICS_GROUP_PROP_REF (pums)
  Mover mean (wtd):        0.287272
  Stayer mean (wtd):       0.281505
  Overall mean (wtd):      0.281866
  Overall SD (wtd):        0.124350
  Diff (Movers - Stayers): 0.005768
  SE:                      0.000610
  95% CI:                  [0.004572, 0.006963]
  Diff / SD:               +0.046382



In [33]:
# proportion_same_naics_* (reference person)
result = compare_means(
    movers[movers["NAICS_AGR_EXT_REF"] == 1],
    stayers[stayers["NAICS_AGR_EXT_REF"] == 1],
    "OWN_NAICS_GROUP_PROP_REF",
    "pums",
)
print_result(result)

OWN_NAICS_GROUP_PROP_REF (pums)
  Mover mean (wtd):        0.034796
  Stayer mean (wtd):       0.054597
  Overall mean (wtd):      0.053594
  Overall SD (wtd):        0.067239
  Diff (Movers - Stayers): -0.019801
  SE:                      0.002046
  95% CI:                  [-0.023811, -0.015791]
  Diff / SD:               -0.294489



In [34]:
# proportion_same_race_* / proportion_also_latino (reference person)
result = compare_means(
    movers[~movers["RACE_ETHNICITY_REF"].isin([7, 8, 9])],
    stayers[~stayers["RACE_ETHNICITY_REF"].isin([7, 8, 9])],
    "OWN_RACE_ETH_PROP_REF",
    "pums",
)
print_result(result)

OWN_RACE_ETH_PROP_REF (pums)
  Mover mean (wtd):        0.514140
  Stayer mean (wtd):       0.532392
  Overall mean (wtd):      0.531355
  Overall SD (wtd):        0.275239
  Diff (Movers - Stayers): -0.018252
  SE:                      0.001374
  95% CI:                  [-0.020945, -0.015559]
  Diff / SD:               -0.066314



In [35]:
# jan_avg_temp_c
result = compare_means(
    movers,
    stayers,
    "JAN_AVG_TEMP_C",
    "migpuma",
)
print_result(result)

JAN_AVG_TEMP_C (migpuma)
  Mover mean (wtd):        3.196369
  Stayer mean (wtd):       3.568813
  Overall mean (wtd):      3.547417
  Overall SD (wtd):        7.077384
  Diff (Movers - Stayers): -0.372444
  SE:                      0.037873
  95% CI:                  [-0.446676, -0.298212]
  Diff / SD:               -0.052625



In [36]:
# proportion_ent_jobs / proportion_ent_jobs_18_34 / _35_64
result = compare_means(
    movers,
    stayers,
    "Proportion of entertainment jobs",
    "migpuma",
)
print_result(result)

Proportion of entertainment jobs (migpuma)
  Mover mean (wtd):        0.017963
  Stayer mean (wtd):       0.017797
  Overall mean (wtd):      0.017806
  Overall SD (wtd):        0.010361
  Diff (Movers - Stayers): 0.000167
  SE:                      0.000048
  95% CI:                  [0.000073, 0.000260]
  Diff / SD:               +0.016076



In [37]:
# proportion_ent_jobs / proportion_ent_jobs_18_34 / _35_64
result = compare_means(
    movers,
    stayers,
    "NUM_CHILDREN",
    "pums",
)
print_result(result)

NUM_CHILDREN (pums)
  Mover mean (wtd):        0.277701
  Stayer mean (wtd):       0.395574
  Overall mean (wtd):      0.388803
  Overall SD (wtd):        0.876689
  Diff (Movers - Stayers): -0.117873
  SE:                      0.003192
  95% CI:                  [-0.124129, -0.111618]
  Diff / SD:               -0.134453



In [38]:
# stay_age_18_22
result = compare_means(
    movers,
    stayers,
    "AGE_18_22",
    "pums",
)
print_result(result)

AGE_18_22 (pums)
  Mover mean (wtd):        0.250166
  Stayer mean (wtd):       0.106690
  Overall mean (wtd):      0.114932
  Overall SD (wtd):        0.318940
  Diff (Movers - Stayers): 0.143477
  SE:                      0.001586
  95% CI:                  [0.140369, 0.146585]
  Diff / SD:               +0.449855



In [39]:
# stay_age_23_29
result = compare_means(
    movers,
    stayers,
    "AGE_23_29",
    "pums",
)
print_result(result)

AGE_23_29 (pums)
  Mover mean (wtd):        0.253258
  Stayer mean (wtd):       0.133984
  Overall mean (wtd):      0.140836
  Overall SD (wtd):        0.347852
  Diff (Movers - Stayers): 0.119274
  SE:                      0.001640
  95% CI:                  [0.116061, 0.122488]
  Diff / SD:               +0.342888



In [40]:
# stay_age_30_39
result = compare_means(
    movers,
    stayers,
    "AGE_30_39",
    "pums",
)
print_result(result)

AGE_30_39 (pums)
  Mover mean (wtd):        0.179477
  Stayer mean (wtd):       0.163664
  Overall mean (wtd):      0.164572
  Overall SD (wtd):        0.370794
  Diff (Movers - Stayers): 0.015813
  SE:                      0.001565
  95% CI:                  [0.012745, 0.018882]
  Diff / SD:               +0.042647



In [41]:
# stay_age_40_49
result = compare_means(
    movers,
    stayers,
    "AGE_40_49",
    "pums",
)
print_result(result)

AGE_40_49 (pums)
  Mover mean (wtd):        0.099773
  Stayer mean (wtd):       0.149766
  Overall mean (wtd):      0.146894
  Overall SD (wtd):        0.354000
  Diff (Movers - Stayers): -0.049992
  SE:                      0.001312
  95% CI:                  [-0.052563, -0.047421]
  Diff / SD:               -0.141221



In [42]:
# stay_age_50_64
result = compare_means(
    movers,
    stayers,
    "AGE_50_64",
    "pums",
)
print_result(result)

AGE_50_64 (pums)
  Mover mean (wtd):        0.129088
  Stayer mean (wtd):       0.240058
  Overall mean (wtd):      0.233683
  Overall SD (wtd):        0.423173
  Diff (Movers - Stayers): -0.110970
  SE:                      0.001271
  95% CI:                  [-0.113462, -0.108479]
  Diff / SD:               -0.262233



In [43]:
# proportion_same_age_65_plus
result = compare_means(
    movers,
    stayers,
    "AGE_OVER_65",
    "pums",
)
print_result(result)

AGE_OVER_65 (pums)
  Mover mean (wtd):        0.088237
  Stayer mean (wtd):       0.205839
  Overall mean (wtd):      0.199083
  Overall SD (wtd):        0.399311
  Diff (Movers - Stayers): -0.117602
  SE:                      0.001312
  95% CI:                  [-0.120173, -0.115031]
  Diff / SD:               -0.294512



In [44]:
# stay_child_under_6
result = compare_means(
    movers,
    stayers,
    "CHILD_UNDER_6",
    "pums",
)
print_result(result)

CHILD_UNDER_6 (pums)
  Mover mean (wtd):        0.091104
  Stayer mean (wtd):       0.089802
  Overall mean (wtd):      0.089876
  Overall SD (wtd):        0.286005
  Diff (Movers - Stayers): 0.001303
  SE:                      0.001220
  95% CI:                  [-0.001089, 0.003695]
  Diff / SD:               +0.004555



In [45]:
# stay_child_6_to_17
result = compare_means(
    movers,
    stayers,
    "CHILD_6_TO_17",
    "pums",
)
print_result(result)

CHILD_6_TO_17 (pums)
  Mover mean (wtd):        0.093214
  Stayer mean (wtd):       0.167212
  Overall mean (wtd):      0.162961
  Overall SD (wtd):        0.369330
  Diff (Movers - Stayers): -0.073998
  SE:                      0.001330
  95% CI:                  [-0.076605, -0.071391]
  Diff / SD:               -0.200357



In [46]:
# proportion_hh_with_children_if_have_children
result = compare_means(
    movers,
    stayers,
    "CHILD",
    "pums",
)
print_result(result)

CHILD (pums)
  Mover mean (wtd):        0.152522
  Stayer mean (wtd):       0.214714
  Overall mean (wtd):      0.211141
  Overall SD (wtd):        0.408119
  Diff (Movers - Stayers): -0.062192
  SE:                      0.001690
  95% CI:                  [-0.065503, -0.058880]
  Diff / SD:               -0.152387



In [47]:
# stay_married_more_than_year
movers["LONG_MARRIED_AND_PAIRED"] = (movers["PAIRED_UNIT"] == 1) & (
    movers["MARRIED_MORE_THAN_YEAR"] == 1
)
stayers["LONG_MARRIED_AND_PAIRED"] = (stayers["PAIRED_UNIT"] == 1) & (
    stayers["MARRIED_MORE_THAN_YEAR"] == 1
)
result = compare_means(
    movers,
    stayers,
    "LONG_MARRIED_AND_PAIRED",
    "pums",
)
print_result(result)

LONG_MARRIED_AND_PAIRED (pums)
  Mover mean (wtd):        0.180996
  Stayer mean (wtd):       0.330684
  Overall mean (wtd):      0.322085
  Overall SD (wtd):        0.467275
  Diff (Movers - Stayers): -0.149687
  SE:                      0.001729
  95% CI:                  [-0.153075, -0.146299]
  Diff / SD:               -0.320341



In [48]:
# stay_married_less_than_year
movers["RECENT_MARRIED_AND_PAIRED"] = (movers["PAIRED_UNIT"] == 1) & (
    movers["RECENTLY_MARRIED"] == 1
)
stayers["RECENT_MARRIED_AND_PAIRED"] = (stayers["PAIRED_UNIT"] == 1) & (
    stayers["RECENTLY_MARRIED"] == 1
)
result = compare_means(
    movers,
    stayers,
    "RECENT_MARRIED_AND_PAIRED",
    "pums",
)
print_result(result)

RECENT_MARRIED_AND_PAIRED (pums)
  Mover mean (wtd):        0.020490
  Stayer mean (wtd):       0.009870
  Overall mean (wtd):      0.010480
  Overall SD (wtd):        0.101836
  Diff (Movers - Stayers): 0.010620
  SE:                      0.000576
  95% CI:                  [0.009491, 0.011748]
  Diff / SD:               +0.104281



In [49]:
# stay_recently_divorced_or_widowed
result = compare_means(
    movers,
    stayers,
    "RECENTLY_WIDOWED_OR_DIVORCED",
    "pums",
)
print_result(result)

RECENTLY_WIDOWED_OR_DIVORCED (pums)
  Mover mean (wtd):        0.022439
  Stayer mean (wtd):       0.018408
  Overall mean (wtd):      0.018639
  Overall SD (wtd):        0.135247
  Diff (Movers - Stayers): 0.004031
  SE:                      0.000627
  95% CI:                  [0.002803, 0.005259]
  Diff / SD:               +0.029806



In [50]:
# stay_2work
result = compare_means(
    movers,
    stayers,
    "WORK2",
    "pums",
)
print_result(result)

WORK2 (pums)
  Mover mean (wtd):        0.139757
  Stayer mean (wtd):       0.192908
  Overall mean (wtd):      0.189854
  Overall SD (wtd):        0.392186
  Diff (Movers - Stayers): -0.053151
  SE:                      0.001550
  95% CI:                  [-0.056189, -0.050113]
  Diff / SD:               -0.135525



In [51]:
# stay_single_parent
result = compare_means(
    movers,
    stayers,
    "SINGLE_UNIT_WITH_CHILD",
    "pums",
)
print_result(result)

SINGLE_UNIT_WITH_CHILD (pums)
  Mover mean (wtd):        0.051082
  Stayer mean (wtd):       0.066982
  Overall mean (wtd):      0.066069
  Overall SD (wtd):        0.248402
  Diff (Movers - Stayers): -0.015900
  SE:                      0.000935
  95% CI:                  [-0.017732, -0.014069]
  Diff / SD:               -0.064010



In [52]:
# stay_edu_at_least_bachelors
result = compare_means(
    movers,
    stayers,
    "EDU_BACHELORS_OR_HIGHER",
    "pums",
)
print_result(result)

EDU_BACHELORS_OR_HIGHER (pums)
  Mover mean (wtd):        0.362627
  Stayer mean (wtd):       0.322740
  Overall mean (wtd):      0.325031
  Overall SD (wtd):        0.468386
  Diff (Movers - Stayers): 0.039887
  SE:                      0.002228
  95% CI:                  [0.035520, 0.044254]
  Diff / SD:               +0.085158



In [53]:
# stay_edu_high_no_bachelors
result = compare_means(
    movers,
    stayers,
    "EDU_HIGH_BUT_NOT_BACHELORS",
    "pums",
)
print_result(result)

EDU_HIGH_BUT_NOT_BACHELORS (pums)
  Mover mean (wtd):        0.573092
  Stayer mean (wtd):       0.571816
  Overall mean (wtd):      0.571889
  Overall SD (wtd):        0.494805
  Diff (Movers - Stayers): 0.001276
  SE:                      0.002378
  95% CI:                  [-0.003384, 0.005936]
  Diff / SD:               +0.002579



In [54]:
# stay_edu_high_no_bachelors
result = compare_means(
    movers,
    stayers,
    "EDU_NOHIGH",
    "pums",
)
print_result(result)

EDU_NOHIGH (pums)
  Mover mean (wtd):        0.064281
  Stayer mean (wtd):       0.105444
  Overall mean (wtd):      0.103080
  Overall SD (wtd):        0.304063
  Diff (Movers - Stayers): -0.041163
  SE:                      0.001108
  95% CI:                  [-0.043335, -0.038992]
  Diff / SD:               -0.135377



In [55]:
# stay_in_college
result = compare_means(
    movers,
    stayers,
    "IN_COLLEGE",
    "pums",
)
print_result(result)

IN_COLLEGE (pums)
  Mover mean (wtd):        0.258835
  Stayer mean (wtd):       0.107657
  Overall mean (wtd):      0.116342
  Overall SD (wtd):        0.320634
  Diff (Movers - Stayers): 0.151178
  SE:                      0.001860
  95% CI:                  [0.147531, 0.154824]
  Diff / SD:               +0.471495



In [56]:
# stay_foreign
result = compare_means(
    movers,
    stayers,
    "FOREIGN_BORN",
    "pums",
)
print_result(result)

FOREIGN_BORN (pums)
  Mover mean (wtd):        0.129633
  Stayer mean (wtd):       0.174953
  Overall mean (wtd):      0.172350
  Overall SD (wtd):        0.377684
  Diff (Movers - Stayers): -0.045321
  SE:                      0.001344
  95% CI:                  [-0.047955, -0.042686]
  Diff / SD:               -0.119996



In [57]:
# stay_mil
result = compare_means(
    movers,
    stayers,
    "IN_MILITARY",
    "pums",
)
print_result(result)

IN_MILITARY (pums)
  Mover mean (wtd):        0.025591
  Stayer mean (wtd):       0.003377
  Overall mean (wtd):      0.004653
  Overall SD (wtd):        0.068055
  Diff (Movers - Stayers): 0.022214
  SE:                      0.000593
  95% CI:                  [0.021052, 0.023377]
  Diff / SD:               +0.326418



In [58]:
# stay_T34
result = compare_means(
    movers,
    stayers,
    "IS_T34",
    "migpuma",
)
print_result(result)

IS_T34 (migpuma)
  Mover mean (wtd):        0.473220
  Stayer mean (wtd):       0.481581
  Overall mean (wtd):      0.481101
  Overall SD (wtd):        0.499643
  Diff (Movers - Stayers): -0.008361
  SE:                      0.002364
  95% CI:                  [-0.012994, -0.003728]
  Diff / SD:               -0.016734



In [59]:
# stay_metro
result = compare_means(
    movers,
    stayers,
    "IS_METRO",
    "migpuma",
)
print_result(result)

IS_METRO (migpuma)
  Mover mean (wtd):        0.378224
  Stayer mean (wtd):       0.362975
  Overall mean (wtd):      0.363851
  Overall SD (wtd):        0.481107
  Diff (Movers - Stayers): 0.015249
  SE:                      0.002329
  95% CI:                  [0.010683, 0.019814]
  Diff / SD:               +0.031695



In [60]:
# stay_metro
result = compare_means(
    movers,
    stayers,
    "NOT_MSA",
    "migpuma",
)
print_result(result)

NOT_MSA (migpuma)
  Mover mean (wtd):        0.148556
  Stayer mean (wtd):       0.155443
  Overall mean (wtd):      0.155048
  Overall SD (wtd):        0.361950
  Diff (Movers - Stayers): -0.006887
  SE:                      0.001768
  95% CI:                  [-0.010353, -0.003422]
  Diff / SD:               -0.019029



In [ ]:
# Everything the destination comparisons below need, built once.
puma_data["IS_T34"] = (puma_data["TYPE_NUM"] == 0).astype(int)
puma_data["IS_METRO"] = (puma_data["TYPE_NUM"] == 1).astype(int)
puma_data["NOT_METRO"] = (puma_data["TYPE_NUM"] == 2).astype(int)
puma_population = puma_data["TOT_POP"].to_numpy(float)


def puma_mean_sd(column):
    """TOT_POP-weighted mean and SD of a puma_data column over all PUMAs."""
    universe = puma_data[column].to_numpy(float)
    present = np.isfinite(universe)
    mean = np.average(universe[present], weights=puma_population[present])
    sd = np.sqrt(
        np.average((universe[present] - mean) ** 2, weights=puma_population[present])
    )
    return mean, sd


def add_own_column(name, column_names):
    """Build `name`, `name`_ALL_PUMA and `name`_ALL_PUMA_SD on `movers` for a quantity
    with no single puma_data column -- the mover's own race category / NAICS group /
    education bracket / age bracket. `column_names` gives the puma_data column that
    applies to each mover; the value is read at their CHOSEN puma and the baseline is
    that same column's all-PUMA mean.
    """
    names = pd.Series(np.asarray(column_names), index=movers.index).replace("", np.nan)
    used = list(names.dropna().unique())
    block = puma_data[used].reindex(movers["CHOSEN"])
    position = block.columns.get_indexer(names.fillna(used[0]))
    movers[name] = np.where(
        names.isna(), np.nan, block.to_numpy(float)[np.arange(len(movers)), position]
    )
    stats = {column: puma_mean_sd(column) for column in used}
    movers[name + "_ALL_PUMA"] = names.map(
        {c: m for c, (m, _) in stats.items()}
    ).to_numpy(float)
    movers[name + "_ALL_PUMA_SD"] = names.map(
        {c: s for c, (_, s) in stats.items()}
    ).to_numpy(float)


AGE_BRACKET_COLS = [
    "Proportion of people 18-34",
    "Proportion of people 35-64",
    "Proportion of people 65+",
]
add_own_column(
    "OWN_AGE_PROP",
    np.select(
        [movers[f].to_numpy() == 1 for f in ["AGE_18_34", "AGE_35_64", "AGE_OVER_65"]],
        AGE_BRACKET_COLS,
        default="",
    ),
)
add_own_column(
    "OWN_EARNINGS_10K_BY_EDU",
    np.select(
        [movers[flag].to_numpy() == 1 for flag in lc.EDU_EARNINGS_RAW_COLS],
        list(lc.EDU_EARNINGS_RAW_COLS.values()),
        default="",
    ),
)
add_own_column("OWN_RACE_ETH_PROP", movers["RACE_ETHNICITY_REF"].map(lc.RACE_COLUMNS))
add_own_column(
    "OWN_NAICS_GROUP_PROP",
    movers["INDNAICS_REF"]
    .str[0:2]
    .map(lc.NAICS_SECTOR_PREFIXES)
    .where(~movers["IN_MILITARY_REF"].astype(bool))
    .map({s: f"NAICS_GROUP_PROP_{g}" for g, ss in lc.NAICS_GROUPS.items() for s in ss}),
)

# distance is pairwise, so the baseline is the population-weighted mean log distance
# from that mover's own ORIGIN to every PUMA -- one value per MIGPUMA row of the matrix
log_distance = np.log1p(distance_matrix)
distance_share = puma_data["TOT_POP"].reindex(log_distance.columns).to_numpy(float)
assert np.isfinite(distance_share).all(), (
    "distance matrix has a PUMA missing from puma_data"
)
distance_share = distance_share / distance_share.sum()
origin_mean_log_distance = log_distance.to_numpy() @ distance_share
origin_var_log_distance = (log_distance.to_numpy() ** 2) @ distance_share - (
    origin_mean_log_distance**2
)
origin_row = log_distance.index.get_indexer(movers["ORIGIN"])
chosen_col = log_distance.columns.get_indexer(movers["CHOSEN"])
assert (origin_row >= 0).all() and (chosen_col >= 0).all(), "missing from the matrix"
movers["LOG_DISTANCE"] = log_distance.to_numpy()[origin_row, chosen_col]
movers["LOG_DISTANCE_ALL_PUMA"] = origin_mean_log_distance[origin_row]
movers["LOG_DISTANCE_ALL_PUMA_SD"] = np.sqrt(origin_var_log_distance[origin_row])

In [62]:
# chosen-PUMA mean vs the mean over all PUMAs, one column at a time.
# Subset `movers` to apply a spec's interaction factor, same as compare_means above.
def compare_chosen_to_all(movers, column):
    """Weighted mean of the puma_data `column` at each mover's CHOSEN puma vs its
    TOT_POP-weighted mean over all PUMAs -- the PUMA the average person already lives in,
    not the average PUMA. The universe is fixed, so only the chosen side carries sampling
    error and the replicate SE is just that of the chosen mean.
    """
    values = puma_data[column].reindex(movers["CHOSEN"]).to_numpy(float)
    weights = movers[WEIGHT_COLUMNS].to_numpy(float)
    keep = np.isfinite(values)
    values, weights = values[keep], weights[keep]

    all_puma_mean, all_puma_sd = puma_mean_sd(column)
    chosen_means = values @ weights / weights.sum(axis=0)
    differences = chosen_means - all_puma_mean

    difference = differences[0]
    standard_error = _replicate_se(differences)
    return {
        "variable": column,
        "chosen_mean": chosen_means[0],
        "all_puma_mean": all_puma_mean,
        "all_puma_sd": all_puma_sd,
        "diff": difference,
        "diff_se": standard_error,
        "diff_ci": (
            difference - 1.96 * standard_error,
            difference + 1.96 * standard_error,
        ),
        "std_diff": difference / all_puma_sd if all_puma_sd > 0 else np.nan,
    }


def compare_own_to_all(movers, column):
    """Same comparison for the `movers` columns built by add_own_column / the distance
    block above, where the baseline varies by mover and so is averaged over movers rather
    than being a constant.
    """
    values = movers[column].to_numpy(float)
    baseline = movers[column + "_ALL_PUMA"].to_numpy(float)
    spread = movers[column + "_ALL_PUMA_SD"].to_numpy(float)
    weights = movers[WEIGHT_COLUMNS].to_numpy(float)

    keep = np.isfinite(values) & np.isfinite(baseline)
    values, baseline, spread, weights = (
        values[keep],
        baseline[keep],
        spread[keep],
        weights[keep],
    )
    total = weights.sum(axis=0)

    chosen_means = values @ weights / total
    baseline_means = baseline @ weights / total
    differences = chosen_means - baseline_means

    difference = differences[0]
    standard_error = _replicate_se(differences)
    all_puma_sd = np.sqrt(np.average(spread**2, weights=weights[:, 0]))
    return {
        "variable": column,
        "chosen_mean": chosen_means[0],
        "all_puma_mean": baseline_means[0],
        "all_puma_sd": all_puma_sd,
        "diff": difference,
        "diff_se": standard_error,
        "diff_ci": (
            difference - 1.96 * standard_error,
            difference + 1.96 * standard_error,
        ),
        "std_diff": difference / all_puma_sd if all_puma_sd > 0 else np.nan,
    }


def print_chosen_result(result):
    low, high = result["diff_ci"]
    print(f"{result['variable']}")
    print(f"  Chosen mean (wtd):       {result['chosen_mean']:,.6f}")
    print(f"  All-PUMA mean (pop-wtd): {result['all_puma_mean']:,.6f}")
    print(f"  All-PUMA SD (pop-wtd):   {result['all_puma_sd']:,.6f}")
    print(f"  Diff (Chosen - All):     {result['diff']:6f}")
    print(f"  SE:                      {result['diff_se']:6f}")
    print(f"  95% CI:                  [{low:e}, {high:6f}]")
    print(f"  Diff / SD:               {result['std_diff']:+.4f}\n")


In [63]:
# log_pop_offset; bit meaningless since this is used for normalization
result = compare_chosen_to_all(
    movers,
    "TOT_POP",
)
print_chosen_result(result)

TOT_POP
  Chosen mean (wtd):       147,783.895321
  All-PUMA mean (pop-wtd): 145,461.256716
  All-PUMA SD (pop-wtd):   32,295.688243
  Diff (Chosen - All):     2322.638604
  SE:                      126.889845
  95% CI:                  [2.073935e+03, 2571.342701]
  Diff / SD:               +0.0719



In [64]:
# destchoice_med_rent_k
result = compare_chosen_to_all(
    movers,
    "Median gross rent in thousands of dollars",
)
print_chosen_result(result)

Median gross rent in thousands of dollars
  Chosen mean (wtd):       1.118761
  All-PUMA mean (pop-wtd): 1.111558
  All-PUMA SD (pop-wtd):   0.380013
  Diff (Chosen - All):     0.007203
  SE:                      0.001556
  95% CI:                  [4.152388e-03, 0.010253]
  Diff / SD:               +0.0190



In [65]:
# destchoice_vacancy_rate
result = compare_chosen_to_all(
    movers,
    "House vacancy proportion",
)
print_chosen_result(result)

House vacancy proportion
  Chosen mean (wtd):       0.116212
  All-PUMA mean (pop-wtd): 0.112344
  All-PUMA SD (pop-wtd):   0.074578
  Diff (Chosen - All):     0.003868
  SE:                      0.000297
  95% CI:                  [3.286383e-03, 0.004449]
  Diff / SD:               +0.0519



In [66]:
# destchoice_unemp
result = compare_chosen_to_all(
    movers,
    "Unemployment rate",
)
print_chosen_result(result)

Unemployment rate
  Chosen mean (wtd):       0.049265
  All-PUMA mean (pop-wtd): 0.050185
  All-PUMA SD (pop-wtd):   0.021252
  Diff (Chosen - All):     -0.000920
  SE:                      0.000087
  95% CI:                  [-1.091110e-03, -0.000749]
  Diff / SD:               -0.0433



In [67]:
# destchoice_alt_commute
result = compare_chosen_to_all(
    movers,
    "Proportion alternative commute",
)
print_chosen_result(result)

Proportion alternative commute
  Chosen mean (wtd):       0.049058
  All-PUMA mean (pop-wtd): 0.041958
  All-PUMA SD (pop-wtd):   0.092189
  Diff (Chosen - All):     0.007100
  SE:                      0.000453
  95% CI:                  [6.212693e-03, 0.007987]
  Diff / SD:               +0.0770



In [68]:
# destchoice_T34
result = compare_chosen_to_all(
    movers,
    "IS_T34",
)
print_chosen_result(result)

IS_T34
  Chosen mean (wtd):       0.447748
  All-PUMA mean (pop-wtd): 0.479588
  All-PUMA SD (pop-wtd):   0.499583
  Diff (Chosen - All):     -0.031840
  SE:                      0.002087
  95% CI:                  [-3.593012e-02, -0.027750]
  Diff / SD:               -0.0637



In [69]:
# destchoice_metro
result = compare_chosen_to_all(
    movers,
    "IS_METRO",
)
print_chosen_result(result)

IS_METRO
  Chosen mean (wtd):       0.412031
  All-PUMA mean (pop-wtd): 0.372392
  All-PUMA SD (pop-wtd):   0.483442
  Diff (Chosen - All):     0.039640
  SE:                      0.001848
  95% CI:                  [3.601718e-02, 0.043262]
  Diff / SD:               +0.0820



In [70]:
# destchoice_metro
result = compare_chosen_to_all(
    movers,
    "NOT_METRO",
)
print_chosen_result(result)

NOT_METRO
  Chosen mean (wtd):       0.140220
  All-PUMA mean (pop-wtd): 0.148020
  All-PUMA SD (pop-wtd):   0.355120
  Diff (Chosen - All):     -0.007800
  SE:                      0.001477
  95% CI:                  [-1.069550e-02, -0.004904]
  Diff / SD:               -0.0220



In [71]:
# median_travel_time
result = compare_chosen_to_all(
    movers,
    "Median travel time",
)
print_chosen_result(result)

Median travel time
  Chosen mean (wtd):       24.309961
  All-PUMA mean (pop-wtd): 25.283578
  All-PUMA SD (pop-wtd):   6.482451
  Diff (Chosen - All):     -0.973617
  SE:                      0.029737
  95% CI:                  [-1.031902e+00, -0.915333]
  Diff / SD:               -0.1502



In [72]:
# jan_avg_temp_c
result = compare_chosen_to_all(
    movers,
    "JAN_AVG_TEMP_C",
)
print_chosen_result(result)

JAN_AVG_TEMP_C
  Chosen mean (wtd):       3.314976
  All-PUMA mean (pop-wtd): 3.465968
  All-PUMA SD (pop-wtd):   7.071633
  Diff (Chosen - All):     -0.150992
  SE:                      0.034444
  95% CI:                  [-2.185021e-01, -0.083482]
  Diff / SD:               -0.0214



In [73]:
# proportion_ent_jobs
result = compare_chosen_to_all(
    movers,
    "Proportion of entertainment jobs",
)
print_chosen_result(result)

Proportion of entertainment jobs
  Chosen mean (wtd):       0.018403
  All-PUMA mean (pop-wtd): 0.017773
  All-PUMA SD (pop-wtd):   0.017644
  Diff (Chosen - All):     0.000629
  SE:                      0.000069
  95% CI:                  [4.942465e-04, 0.000764]
  Diff / SD:               +0.0357



In [74]:
# proportion_hh_with_children_if_have_children
result = compare_chosen_to_all(
    movers[movers["CHILD"] == 1],
    "Proportion of households with children",
)
print_chosen_result(result)

Proportion of households with children
  Chosen mean (wtd):       0.315958
  All-PUMA mean (pop-wtd): 0.306526
  All-PUMA SD (pop-wtd):   0.072149
  Diff (Chosen - All):     0.009432
  SE:                      0.000640
  95% CI:                  [8.177322e-03, 0.010687]
  Diff / SD:               +0.1307



In [75]:
# proportion_college_if_in_college
result = compare_chosen_to_all(
    movers[movers["IN_COLLEGE"] == 1],
    "Proportion of people in college",
)
print_chosen_result(result)

Proportion of people in college
  Chosen mean (wtd):       0.112117
  All-PUMA mean (pop-wtd): 0.067489
  All-PUMA SD (pop-wtd):   0.037277
  Diff (Chosen - All):     0.044628
  SE:                      0.000656
  95% CI:                  [4.334099e-02, 0.045914]
  Diff / SD:               +1.1972



In [76]:
# proportion_foreign_if_foreign
result = compare_chosen_to_all(
    movers[movers["FOREIGN_BORN"] == 1],
    "Proportion foreign born",
)
print_chosen_result(result)

Proportion foreign born
  Chosen mean (wtd):       0.193016
  All-PUMA mean (pop-wtd): 0.136617
  All-PUMA SD (pop-wtd):   0.120962
  Diff (Chosen - All):     0.056399
  SE:                      0.001513
  95% CI:                  [5.343383e-02, 0.059364]
  Diff / SD:               +0.4663



In [77]:
# proportion_also_mil
result = compare_chosen_to_all(
    movers[movers["IN_MILITARY"] == 1],
    "Proportion of people in military",
)
print_chosen_result(result)

Proportion of people in military
  Chosen mean (wtd):       0.092146
  All-PUMA mean (pop-wtd): 0.005728
  All-PUMA SD (pop-wtd):   0.020096
  Diff (Chosen - All):     0.086418
  SE:                      0.002118
  95% CI:                  [8.226591e-02, 0.090570]
  Diff / SD:               +4.3003



In [78]:
# destchoice_logdist
result = compare_own_to_all(
    movers,
    "LOG_DISTANCE",
)
print_chosen_result(result)

LOG_DISTANCE
  Chosen mean (wtd):       12.598196
  All-PUMA mean (pop-wtd): 14.285384
  All-PUMA SD (pop-wtd):   0.806856
  Diff (Chosen - All):     -1.687188
  SE:                      0.006323
  95% CI:                  [-1.699581e+00, -1.674795]
  Diff / SD:               -2.0911



In [79]:
# proportion_same_age_18_34 / _35_64 / _65_plus
result = compare_own_to_all(
    movers,
    "OWN_AGE_PROP",
)
print_chosen_result(result)

OWN_AGE_PROP
  Chosen mean (wtd):       0.292449
  All-PUMA mean (pop-wtd): 0.271827
  All-PUMA SD (pop-wtd):   0.048149
  Diff (Chosen - All):     0.020621
  SE:                      0.000328
  95% CI:                  [1.997859e-02, 0.021264]
  Diff / SD:               +0.4283



In [80]:
# destchoice_med_earnings_10k_no_degree / _degree
result = compare_own_to_all(
    movers,
    "OWN_EARNINGS_10K_BY_EDU",
)
print_chosen_result(result)

OWN_EARNINGS_10K_BY_EDU
  Chosen mean (wtd):       4.456070
  All-PUMA mean (pop-wtd): 4.352355
  All-PUMA SD (pop-wtd):   0.969839
  Diff (Chosen - All):     0.103715
  SE:                      0.004372
  95% CI:                  [9.514592e-02, 0.112284]
  Diff / SD:               +0.1069



In [83]:
# proportion_same_race_* / proportion_also_latino (reference person)
result = compare_own_to_all(
    movers[~movers["RACE_ETHNICITY_REF"].isin([7, 8, 9])],
    "OWN_RACE_ETH_PROP",
)
print_chosen_result(result)

OWN_RACE_ETH_PROP
  Chosen mean (wtd):       0.536630
  All-PUMA mean (pop-wtd): 0.450458
  All-PUMA SD (pop-wtd):   0.224121
  Diff (Chosen - All):     0.086173
  SE:                      0.000944
  95% CI:                  [8.432283e-02, 0.088022]
  Diff / SD:               +0.3845



In [87]:
# proportion_same_naics_* (reference person)
result = compare_own_to_all(
    movers[~movers["OWN_NAICS_GROUP_PROP"].isna()],
    "OWN_NAICS_GROUP_PROP",
)
print_chosen_result(result)

OWN_NAICS_GROUP_PROP
  Chosen mean (wtd):       0.292787
  All-PUMA mean (pop-wtd): 0.278720
  All-PUMA SD (pop-wtd):   0.092443
  Diff (Chosen - All):     0.014067
  SE:                      0.000437
  95% CI:                  [1.321117e-02, 0.014923]
  Diff / SD:               +0.1522

